# 06. External validation — Part 1: preflight and frozen feature contract

## Цель этой части

Перед подключением внешнего датасета мы **ничего не переобучаем** и не меняем результаты `04`.

На этом шаге мы:

1. находим корень проекта;
2. загружаем исходный датасет;
3. восстанавливаем точный контракт входных признаков;
4. проверяем, что у нас действительно 37 разрешённых входных признаков;
5. проверяем сохранённые артефакты frozen ensemble;
6. выясняем, есть ли на диске **реальные обученные модели / calibrators**, которыми можно сделать inference на внешних пациентах;
7. сохраняем feature contract для следующей части.

> Важно: OOF predictions сами по себе недостаточны для внешней валидации.  
> Чтобы прогнать новый внешний датасет, нужны сохранённые обученные модели либо отдельно зафиксированный final inference pipeline.


In [1]:
from pathlib import Path
import json
import sys

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 200)

print("Python:", sys.version.split()[0])
print("Текущая рабочая папка:", Path.cwd())


Python: 3.11.16
Текущая рабочая папка: /home/sinopah/meditron/notebooks


## 1. Находим корень проекта

Ноутбук может быть запущен из `meditron/`, `meditron/notebooks/` или другой вложенной папки.  
Поэтому не привязываемся жёстко к текущей рабочей директории.


In [2]:
def find_project_root(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()

    candidates = [start, *start.parents]

    for candidate in candidates:
        dataset_path = candidate / "data" / "raw" / "deficiency_anemia.csv"
        if dataset_path.exists():
            return candidate

    raise FileNotFoundError(
        "Не удалось найти корень проекта. "
        "Ожидался файл data/raw/deficiency_anemia.csv."
    )


PROJECT_ROOT = find_project_root()

DATA_PATH = PROJECT_ROOT / "data" / "raw" / "deficiency_anemia.csv"
ARTIFACTS_DIR = PROJECT_ROOT / "artifacts"
ENSEMBLE_DIR = ARTIFACTS_DIR / "ensemble"
EXTERNAL_DIR = ARTIFACTS_DIR / "external_validation"

EXTERNAL_DIR.mkdir(parents=True, exist_ok=True)

print("PROJECT_ROOT :", PROJECT_ROOT)
print("DATA_PATH    :", DATA_PATH)
print("ARTIFACTS_DIR:", ARTIFACTS_DIR)
print("ENSEMBLE_DIR :", ENSEMBLE_DIR)


PROJECT_ROOT : /home/sinopah/meditron
DATA_PATH    : /home/sinopah/meditron/data/raw/deficiency_anemia.csv
ARTIFACTS_DIR: /home/sinopah/meditron/artifacts
ENSEMBLE_DIR : /home/sinopah/meditron/artifacts/ensemble


## 2. Загружаем исходный датасет и строим frozen feature contract

Мы исключаем:

- `patient_id`;
- все target / derived columns, которые нельзя подавать модели.

Список должен дать **ровно 37 входных признаков**.


In [3]:
df = pd.read_csv(DATA_PATH)

TARGET_COLUMNS = [
    "anemia",
    "iron_deficiency",
    "B12_deficiency",
    "folate_deficiency",
    "B6_deficiency",
    "copper_deficiency",
    "inflammation_anemia",
    "mixed_deficiency",
    "anemia_class",
    "deficiency_cause",
]

ID_COLUMNS = ["patient_id"]

EXCLUDED_COLUMNS = ID_COLUMNS + TARGET_COLUMNS

missing_expected_columns = [
    col for col in EXCLUDED_COLUMNS
    if col not in df.columns
]

if missing_expected_columns:
    raise ValueError(
        "В исходном датасете отсутствуют ожидаемые служебные/target колонки: "
        f"{missing_expected_columns}"
    )

FEATURE_COLUMNS = [
    col for col in df.columns
    if col not in EXCLUDED_COLUMNS
]

print("Размер исходного датасета:", df.shape)
print("Количество всех колонок:", len(df.columns))
print("Исключено колонок:", len(EXCLUDED_COLUMNS))
print("Количество разрешённых input features:", len(FEATURE_COLUMNS))

assert len(FEATURE_COLUMNS) == 37, (
    "Ожидалось 37 разрешённых признаков, "
    f"но найдено {len(FEATURE_COLUMNS)}. "
    "Не продолжаем, пока не разберёмся с контрактом."
)

print("\nFeature contract подтверждён: 37 признаков.")


Размер исходного датасета: (840, 48)
Количество всех колонок: 48
Исключено колонок: 11
Количество разрешённых input features: 37

Feature contract подтверждён: 37 признаков.


In [4]:
feature_contract = pd.DataFrame({
    "feature": FEATURE_COLUMNS,
    "dtype_internal": [str(df[col].dtype) for col in FEATURE_COLUMNS],
    "missing_n": [int(df[col].isna().sum()) for col in FEATURE_COLUMNS],
    "missing_pct": [
        round(float(df[col].isna().mean() * 100), 2)
        for col in FEATURE_COLUMNS
    ],
    "non_null_n": [int(df[col].notna().sum()) for col in FEATURE_COLUMNS],
})

feature_contract


,feature,dtype_internal,missing_n,missing_pct,non_null_n
0,age_years,int64,0,0.00,840
1,sex,str,0,0.00,840
2,hemoglobin,float64,0,0.00,840
3,RBC,float64,0,0.00,840
4,hematocrit,float64,0,0.00,840
5,MCV,float64,0,0.00,840
6,MCH,float64,0,0.00,840
7,MCHC,float64,0,0.00,840
8,RDW,float64,15,1.79,825
9,platelets,int64,0,0.00,840


## 3. Проверяем frozen ensemble artifacts

Сначала смотрим уже сохранённые табличные артефакты из `04`.


In [5]:
expected_table_artifacts = [
    "ensemble_oof_predictions.csv",
    "ensemble_meta_settings.csv",
    "calibration_summary.csv",
    "model_comparison.csv",
    "binary_ensemble_summary.csv",
]

artifact_status = []

for name in expected_table_artifacts:
    path = ENSEMBLE_DIR / name
    artifact_status.append({
        "artifact": name,
        "exists": path.exists(),
        "path": str(path),
        "size_kb": round(path.stat().st_size / 1024, 2) if path.exists() else np.nan,
    })

artifact_status = pd.DataFrame(artifact_status)
artifact_status


,artifact,exists,path,size_kb
0,ensemble_oof_predictions.csv,True,/home/sinopah/meditron/artifacts/ensemble/ense...,220.09
1,ensemble_meta_settings.csv,True,/home/sinopah/meditron/artifacts/ensemble/ense...,2.61
2,calibration_summary.csv,True,/home/sinopah/meditron/artifacts/ensemble/cali...,3.21
3,model_comparison.csv,True,/home/sinopah/meditron/artifacts/ensemble/mode...,0.34
4,binary_ensemble_summary.csv,True,/home/sinopah/meditron/artifacts/ensemble/bina...,0.97


In [6]:
oof_path = ENSEMBLE_DIR / "ensemble_oof_predictions.csv"
meta_path = ENSEMBLE_DIR / "ensemble_meta_settings.csv"

if oof_path.exists():
    oof = pd.read_csv(oof_path)
    print("OOF shape:", oof.shape)
    display(oof.head(3))
else:
    print("ensemble_oof_predictions.csv не найден.")

if meta_path.exists():
    ensemble_meta = pd.read_csv(meta_path)
    print("\nensemble_meta_settings.csv:")
    display(ensemble_meta)
else:
    print("ensemble_meta_settings.csv не найден.")


OOF shape: (840, 30)


,patient_id,anemia_class_true,deficiency_cause_true,anemia_rule,ensemble_class_pred,ensemble_cause_pred,iron_deficiency__true,iron_deficiency__probability,iron_deficiency__prediction,iron_deficiency__rule_score,B12_deficiency__true,B12_deficiency__probability,B12_deficiency__prediction,B12_deficiency__rule_score,folate_deficiency__true,folate_deficiency__probability,folate_deficiency__prediction,folate_deficiency__rule_score,B6_deficiency__true,B6_deficiency__probability,B6_deficiency__prediction,B6_deficiency__rule_score,copper_deficiency__true,copper_deficiency__probability,copper_deficiency__prediction,copper_deficiency__rule_score,inflammation_anemia__true,inflammation_anemia__probability,inflammation_anemia__prediction,inflammation_anemia__rule_score
0,P000001,B12_deficiency_no_anemia,B12_deficiency,0,B12_deficiency_no_anemia,B12_deficiency,0,0.000889,0,0.050,1,0.828236,1,0.15,0,0.130243,0,0.95,0,0.003322,0,0.5,0,0.000654,0,0.5,0,0.000183,0,0.05
1,P000002,folate_deficiency_no_anemia,folate_deficiency,0,no_anemia_no_deficiency,none,0,0.000054,0,0.050,0,0.011015,0,0.50,1,0.081216,0,0.50,0,0.000748,0,0.5,0,0.002415,0,0.5,0,0.000186,0,0.05
2,P000003,latent_deficiency,iron_deficiency,0,latent_deficiency,iron_deficiency,1,0.999953,1,0.725,0,0.000044,0,0.15,0,0.022020,0,0.50,0,0.000736,0,0.5,0,0.000709,0,0.5,0,0.000108,0,0.05



ensemble_meta_settings.csv:


,target,fold,w_l1,w_cb,w_rule,brier,threshold,train_f1_at_threshold
0,iron_deficiency,1,0.4,0.6,0,0.008309,0.41,0.989154
1,iron_deficiency,2,0.5,0.5,0,0.005232,0.40,0.993492
2,iron_deficiency,3,0.4,0.6,0,0.008164,0.42,0.989154
3,iron_deficiency,4,0.3,0.7,0,0.005798,0.48,0.993492
4,iron_deficiency,5,0.4,0.6,0,0.005268,0.33,0.993521
5,B12_deficiency,1,0.5,0.5,0,0.009698,0.66,0.976744
6,B12_deficiency,2,0.4,0.6,0,0.010754,0.26,0.973856
7,B12_deficiency,3,0.5,0.5,0,0.009794,0.30,0.980263
8,B12_deficiency,4,0.5,0.5,0,0.012189,0.26,0.970684
9,B12_deficiency,5,0.6,0.4,0,0.007857,0.23,0.983607


## 4. Проверяем наличие deployable model artifacts

Для внешнего датасета недостаточно иметь только OOF probabilities.

Нужно найти сериализованные модели / calibrators / pipeline, например:

- `.joblib`
- `.pkl`
- `.pickle`
- `.cbm`

Если таких объектов нет, **не будем притворяться, что внешний inference уже возможен**.  
Тогда следующей частью сначала корректно зафиксируем production inference bundle, не меняя OOF-результаты `04`.


In [7]:
MODEL_SUFFIXES = {
    ".joblib",
    ".pkl",
    ".pickle",
    ".cbm",
}

model_artifacts = []

if ARTIFACTS_DIR.exists():
    for path in ARTIFACTS_DIR.rglob("*"):
        if path.is_file() and path.suffix.lower() in MODEL_SUFFIXES:
            model_artifacts.append({
                "name": path.name,
                "suffix": path.suffix.lower(),
                "relative_path": str(path.relative_to(PROJECT_ROOT)),
                "size_kb": round(path.stat().st_size / 1024, 2),
            })

model_artifacts = pd.DataFrame(model_artifacts)

if len(model_artifacts) == 0:
    print("Сериализованные обученные модели / calibrators НЕ найдены.")
else:
    print(f"Найдено model artifacts: {len(model_artifacts)}")
    display(model_artifacts)


Сериализованные обученные модели / calibrators НЕ найдены.


## 5. Проверяем feature-related metadata

Иногда feature list, model config или thresholds уже сохранены в JSON/CSV/TXT.  
Выведем список небольших файлов в `artifacts/ensemble`, чтобы понять, что именно у нас зафиксировано.


In [8]:
ensemble_files = []

if ENSEMBLE_DIR.exists():
    for path in sorted(ENSEMBLE_DIR.rglob("*")):
        if path.is_file():
            ensemble_files.append({
                "name": path.name,
                "suffix": path.suffix.lower(),
                "size_kb": round(path.stat().st_size / 1024, 2),
                "relative_path": str(path.relative_to(PROJECT_ROOT)),
            })

ensemble_files = pd.DataFrame(ensemble_files)

print("Всего файлов в artifacts/ensemble:", len(ensemble_files))
ensemble_files


Всего файлов в artifacts/ensemble: 5


,name,suffix,size_kb,relative_path
0,binary_ensemble_summary.csv,.csv,0.97,artifacts/ensemble/binary_ensemble_summary.csv
1,calibration_summary.csv,.csv,3.21,artifacts/ensemble/calibration_summary.csv
2,ensemble_meta_settings.csv,.csv,2.61,artifacts/ensemble/ensemble_meta_settings.csv
3,ensemble_oof_predictions.csv,.csv,220.09,artifacts/ensemble/ensemble_oof_predictions.csv
4,model_comparison.csv,.csv,0.34,artifacts/ensemble/model_comparison.csv


## 6. Сохраняем frozen feature contract

Этот файл будет использоваться в следующей части для сопоставления колонок внешнего датасета с нашими 37 признаками.


In [9]:
feature_contract_path = EXTERNAL_DIR / "internal_feature_contract.csv"

feature_contract.to_csv(
    feature_contract_path,
    index=False,
)

print("Сохранено:", feature_contract_path)


Сохранено: /home/sinopah/meditron/artifacts/external_validation/internal_feature_contract.csv


## 7. Итог preflight

`ready_for_external_inference = True` только если:

- feature contract корректен;
- есть OOF/meta artifacts;
- на диске найдены реальные сериализованные model artifacts.

Если `False`, это **не ошибка модели**. Это означает, что перед внешней валидацией нам нужно отдельно упаковать уже выбранную архитектуру в frozen inference bundle.


In [10]:
feature_contract_ok = len(FEATURE_COLUMNS) == 37

oof_exists = oof_path.exists()
meta_exists = meta_path.exists()
deployable_models_found = len(model_artifacts) > 0

ready_for_external_inference = (
    feature_contract_ok
    and oof_exists
    and meta_exists
    and deployable_models_found
)

preflight_summary = pd.DataFrame([
    {
        "check": "internal_dataset_loaded",
        "ok": DATA_PATH.exists(),
        "details": f"shape={df.shape}",
    },
    {
        "check": "feature_contract_37",
        "ok": feature_contract_ok,
        "details": f"n_features={len(FEATURE_COLUMNS)}",
    },
    {
        "check": "ensemble_oof_predictions",
        "ok": oof_exists,
        "details": str(oof_path),
    },
    {
        "check": "ensemble_meta_settings",
        "ok": meta_exists,
        "details": str(meta_path),
    },
    {
        "check": "deployable_model_artifacts",
        "ok": deployable_models_found,
        "details": f"n_model_artifacts={len(model_artifacts)}",
    },
    {
        "check": "READY_FOR_EXTERNAL_INFERENCE",
        "ok": ready_for_external_inference,
        "details": (
            "Можно переходить к mapping внешнего датасета."
            if ready_for_external_inference
            else "Сначала нужно проверить/создать frozen inference bundle."
        ),
    },
])

display(preflight_summary)

preflight_summary.to_csv(
    EXTERNAL_DIR / "preflight_summary.csv",
    index=False,
)

print("\nREADY_FOR_EXTERNAL_INFERENCE =", ready_for_external_inference)


,check,ok,details
0,internal_dataset_loaded,True,"shape=(840, 48)"
1,feature_contract_37,True,n_features=37
2,ensemble_oof_predictions,True,/home/sinopah/meditron/artifacts/ensemble/ense...
3,ensemble_meta_settings,True,/home/sinopah/meditron/artifacts/ensemble/ense...
4,deployable_model_artifacts,False,n_model_artifacts=0
5,READY_FOR_EXTERNAL_INFERENCE,False,Сначала нужно проверить/создать frozen inferen...



READY_FOR_EXTERNAL_INFERENCE = False


# Что прислать после запуска этой части

Нужны три вывода:

1. строка  
   `Feature contract подтверждён: 37 признаков.`

2. таблица `model_artifacts` или сообщение  
   `Сериализованные обученные модели / calibrators НЕ найдены.`

3. итог  
   `READY_FOR_EXTERNAL_INFERENCE = True/False`

После этого решим **следующий шаг без гаданий**:

- если `True` → подключаем внешний датасет и строим реальное пересечение признаков;
- если `False` → сначала упаковываем frozen inference bundle из уже выбранных L1 + CatBoost + calibration + ensemble settings, не трогая OOF-метрики `04`.
